# 📋 AI 契約條款自動審查教學
## 用大語言模型幫 B 公司法務部門檢查契約

---

### 🎯 學習目標
透過這份教材，你將學會：
1. 從網路自動下載契約範本與當事人資料
2. 呼叫 AI 模型，依照 5 條審查規則逐條分析契約
3. 輸出一份完整的 **契約審查分析報告**

---

### 🤖 支援的 AI 模型（三選一）

| 模型 | 費用 | 申請網址 |
|------|------|---------|
| 🟢 **Gemini**（Google） | 免費額度，每天1,500次 | https://aistudio.google.com/ |
| 🟡 **GPT**（OpenAI） | 新帳號有試用金 | https://platform.openai.com/ |
| 🔵 **Claude**（Anthropic） | 需付費 | https://console.anthropic.com/ |

> 💡 **學生推薦使用 Gemini**，用 Google 帳號登入即可，完全免費！

---

### 📋 本教材使用的 5 條審查規則

| 編號 | 審查重點 | 說明 |
|------|---------|------|
| 規則1 | 付款條件公平性 | 頭期款比例是否超過30%？付款期限是否合理？ |
| 規則2 | 違約金是否過重 | 逾期違約金比例是否超過千分之5？ |
| 規則3 | 保密期間合理性 | 保密義務期間是否超過5年？ |
| 規則4 | 智慧財產權歸屬 | 是否有一方完全壟斷所有IP權利？ |
| 規則5 | 終止條款平衡性 | 是否只有單方（甲方）才能終止契約？ |


## Step 1｜安裝必要套件

**這格在做什麼**：安裝三個 AI 模型的 Python 套件。在 Colab 每次開啟都需執行一次。

In [ ]:
!pip install anthropic openai google-genai ipywidgets -q

print("✅ 套件安裝完成！")
print("   已安裝：anthropic（Claude）、openai（GPT）、google-genai（Gemini）")

## Step 2｜選擇 AI 模型並填入金鑰

**這格在做什麼**：執行後會出現互動式介面，讓你選擇 AI 模型並輸入金鑰。輸入時內容會顯示為黑點（●●●●），不會洩漏。

| 模型 | 費用 | 申請網址 |
|------|------|---------|
| 🟢 **Gemini** | ✅ 免費額度 | https://aistudio.google.com/ |
| 🟡 **GPT** | 新帳號有試用金 | https://platform.openai.com/ |
| 🔵 **Claude** | 需付費 | https://console.anthropic.com/ |

> ⚠️ 選好模型、填好金鑰後，請點「**✅ 確認送出**」按鈕，才算完成設定。


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── 下拉選單：選擇 AI 模型 ──────────────────────
model_dropdown = widgets.Dropdown(
    options=[
        ("🟢 Gemini（免費，學生推薦）", "gemini"),
        ("🟡 GPT（OpenAI）", "gpt"),
        ("🔵 Claude（Anthropic）", "claude"),
    ],
    value="gemini",
    description="AI 模型：",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="340px")
)

# ── 密碼輸入框：輸入時顯示黑點 ─────────────────
hint_label = widgets.Label(value="🔑 請在下方貼上 API 金鑰（輸入時顯示為 ●●●●）：")

key_input = widgets.Password(
    placeholder="貼上你的 API 金鑰",
    layout=widgets.Layout(width="420px")
)

# ── 確認按鈕 ────────────────────────────────────
confirm_btn = widgets.Button(
    description="✅ 確認送出",
    button_style="success",
    layout=widgets.Layout(width="130px")
)

output_area = widgets.Output()

# ── 按下確認後的動作 ────────────────────────────
def on_confirm(b):
    global AI_MODEL, API_KEY
    AI_MODEL = model_dropdown.value
    API_KEY  = key_input.value
    with output_area:
        clear_output()
        if not API_KEY or len(API_KEY) < 10:
            print("❌ 金鑰好像沒有填入，請在輸入框貼上金鑰後再按確認")
            return
        model_label = {
            "gemini": "🟢 Gemini",
            "gpt":    "🟡 GPT",
            "claude": "🔵 Claude"
        }[AI_MODEL]
        masked = API_KEY[:8] + "******"
        print(f"✅ 模型：{model_label}")
        print(f"✅ 金鑰已接收（開頭：{masked}）")
        print("✅ 設定完成，請繼續執行下一格（Step 3）")

confirm_btn.on_click(on_confirm)

# ── 顯示完整介面 ────────────────────────────────
display(widgets.VBox([
    model_dropdown,
    hint_label,
    key_input,
    confirm_btn,
    output_area
]))


## Step 3｜載入套件與初始化 AI

**這格在做什麼**：載入所有需要的工具，根據你選擇的模型初始化 AI 用戶端，並**自動偵測目前是從哪個 GitHub repo 開啟**，確保資料來源正確。

> 💡 不管你是直接使用老師的版本，還是 fork 之後自己修改過，程式都會自動抓到正確的資料。


In [ ]:
import json, re, requests, os
from datetime import datetime

# ── 自動偵測 BASE_URL（支援 fork）────────────────────
# 改用 COLAB_NOTEBOOK_URL 環境變數，比 _message 更穩定
try:
    notebook_url = os.environ.get("COLAB_NOTEBOOK_URL", "")

    if not notebook_url:
        # 備用方案：從 colab backend 取得
        from google.colab import _message
        result = _message.blocking_request("get_ipynb_url", timeout_sec=5)
        if isinstance(result, dict):
            notebook_url = str(list(result.values())[0] if result else "")
        else:
            notebook_url = str(result or "")

    match = re.search(
        r"github\.com/([^/]+)/([^/]+)/blob/([^/]+)/",
        notebook_url
    )
    if match:
        gh_user   = match.group(1)
        gh_repo   = match.group(2)
        gh_branch = match.group(3)
        BASE_URL = f"https://raw.githubusercontent.com/{gh_user}/{gh_repo}/{gh_branch}"
        print(f"✅ 自動偵測成功！")
        print(f"   GitHub 帳號：{gh_user}  Repo：{gh_repo}  Branch：{gh_branch}")
    else:
        raise ValueError(f"網址中找不到 repo 資訊")

except Exception as e:
    BASE_URL = "https://raw.githubusercontent.com/mjib007/contract-review/main"
    print(f"⚠️  自動偵測未成功，使用預設來源：mjib007/contract-review")

print(f"   資料來源：{BASE_URL}")
print()

# ── 初始化 AI 用戶端 ──────────────────────────────────
if AI_MODEL == "gemini":
    from google import genai as google_genai
    gemini_client = google_genai.Client(api_key=API_KEY)
    GEMINI_MODEL  = "gemini-2.5-flash-preview-05-20"
    print(f"✅ Gemini 初始化完成！（使用 {GEMINI_MODEL}）")

elif AI_MODEL == "gpt":
    from openai import OpenAI
    gpt_client = OpenAI(api_key=API_KEY)
    print("✅ GPT 初始化完成！（使用 gpt-4o-mini）")

elif AI_MODEL == "claude":
    import anthropic
    claude_client = anthropic.Anthropic(api_key=API_KEY)
    print("✅ Claude 初始化完成！（使用 claude-sonnet-4-20250514）")

print(f"   執行時間：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## Step 4｜從 GitHub 下載契約範本與當事人資料

**這格在做什麼**：自動從 GitHub 下載兩個檔案：
- `contract_template.txt`：契約範本（含佔位符）
- `party_data.json`：當事人基本資料

> 💡 老師可以直接修改 GitHub 上的 `party_data.json`，換成不同的教學情境！


In [ ]:
r1 = requests.get(f"{BASE_URL}/contract_template.txt")
r1.encoding = "utf-8"
template = r1.text
print(f"✅ 契約範本下載成功（{len(template)} 字元）")

r2 = requests.get(f"{BASE_URL}/party_data.json")
party_data = r2.json()
print(f"✅ 當事人資料下載成功")
print()
print(f"甲方：{party_data['PARTY_A_NAME']}（代表人：{party_data['PARTY_A_REPRESENTATIVE']}）")
print(f"乙方：{party_data['PARTY_B_NAME']}（代表人：{party_data['PARTY_B_REPRESENTATIVE']}）")
print()
print(f"契約總價：NT$ {party_data['TOTAL_AMOUNT']} 元")
print(f"頭期款：NT$ {party_data['DEPOSIT_AMOUNT']} 元")
print(f"違約金比例：千分之 {party_data['PENALTY_RATIO']}")
print(f"保密期間：{party_data['CONFIDENTIALITY_YEARS']} 年")

## Step 5｜自動填充契約範本

**這格在做什麼**：把所有 `{{佔位符}}` 替換成 JSON 資料中對應的真實資料，產生完整契約文字。

In [ ]:
filled_contract = template
for key, value in party_data.items():
    placeholder = "{{" + key + "}}"
    filled_contract = filled_contract.replace(placeholder, str(value))

remaining = re.findall(r'\{\{.*?\}\}', filled_contract)
if remaining:
    print(f"⚠️  注意：以下佔位符未被填充：{remaining}")
else:
    print("✅ 所有佔位符填充完成，無遺漏！")

print()
print("📄 已填充契約預覽（前400字）：")
print("─" * 50)
print(filled_contract[:400])
print("─" * 50)

## Step 6｜定義審查規則與 AI 審查函式

**這格在做什麼**：設定 B 公司的 5 條審查規則，以及依照你選擇的模型送交審查的函式。

> 💡 修改下方 `REVIEW_RULES` 的內容，就能換成你自己設計的審查規則！


In [ ]:
REVIEW_RULES = [
    {"id": 1, "name": "付款條件公平性",
     "description": "頭期款比例是否超過契約總金額的30%？付款期限是否超過60個工作日？如有異常，請指出條文位置與風險。"},
    {"id": 2, "name": "違約金是否過重",
     "description": "每日逾期違約金的比例是否超過契約總價款的千分之5？過高的違約金對乙方（B公司）不利，請評估風險。"},
    {"id": 3, "name": "保密期間合理性",
     "description": "保密義務的期間是否超過5年？超過5年的保密義務在實務上可能過於嚴苛，請指出並提供修改建議。"},
    {"id": 4, "name": "智慧財產權歸屬",
     "description": "智慧財產權是否完全歸甲方所有，乙方是否毫無保留任何權利？請分析並建議修改方向。"},
    {"id": 5, "name": "終止條款平衡性",
     "description": "契約終止權是否只賦予甲方（A公司），乙方是否有對等的終止權？若雙方權利不對等，請指出並建議修改方式。"}
]

def build_prompt(contract_text, rule):
    return f"""你是一位專業的企業法務顧問，正在協助 B 公司審查一份由 A 公司提供的技術服務契約。

請依照以下審查規則，仔細分析契約內容：

【審查規則 {rule['id']}：{rule['name']}】
{rule['description']}

【待審查契約全文】
{contract_text}

請以以下格式回應：
1. **審查結果**：✅ 無問題 / ⚠️ 有疑慮 / ❌ 建議修改
2. **問題條文**：指出有問題的條文編號與內容（若無問題則填「無」）
3. **風險說明**：說明為何構成問題、對 B 公司的潛在風險
4. **修改建議**：具體建議如何修改條文內容（若無問題則填「無需修改」）

請用繁體中文回應，語氣專業但易懂。"""

def review_contract_with_ai(contract_text, rule):
    prompt = build_prompt(contract_text, rule)

    if AI_MODEL == "gemini":
        from google import genai as google_genai
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt
        )
        return response.text

    elif AI_MODEL == "gpt":
        response = gpt_client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens=1000,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

    elif AI_MODEL == "claude":
        response = claude_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1000,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

print(f"✅ 審查規則設定完成！（使用模型：{AI_MODEL.upper()}）")
for rule in REVIEW_RULES:
    print(f"   規則{rule['id']}：{rule['name']}")


## Step 7｜執行自動審查

**這格在做什麼**：依序將契約送交 AI，按照 5 條規則逐一審查，每條完成後即時顯示結果。

> ⚠️ 這個步驟會呼叫 AI API（約需 30–60 秒），請耐心等候。


In [ ]:
print(f"🔍 開始進行契約自動審查（模型：{AI_MODEL.upper()}）...")
print("=" * 60)
print()

review_results = []

for rule in REVIEW_RULES:
    print(f"📌 正在執行 規則{rule['id']}：{rule['name']}...")
    result = review_contract_with_ai(filled_contract, rule)
    review_results.append({"rule_id": rule['id'], "rule_name": rule['name'], "result": result})
    print(f"\n{'-' * 50}")
    print(f"【規則{rule['id']}：{rule['name']}】審查結果")
    print('-' * 50)
    print(result)
    print()

print("=" * 60)
print(f"✅ 全部 {len(REVIEW_RULES)} 條規則審查完成！")

## Step 8｜輸出完整審查報告

**這格在做什麼**：將所有審查結果整合成一份正式報告，並儲存為文字檔。

> 💡 執行完成後，點選左側資料夾圖示，找到報告檔案即可下載。

In [ ]:
lines = []
lines.append("=" * 60)
lines.append("        契約條款審查分析報告")
lines.append("=" * 60)
lines.append(f"審查日期：{datetime.now().strftime("%Y年%m月%d日 %H:%M")}")
lines.append(f"使用模型：{AI_MODEL.upper()}")
lines.append(f"甲方：{party_data['PARTY_A_NAME']}")
lines.append(f"乙方：{party_data['PARTY_B_NAME']}")
lines.append(f"契約標的：{party_data['SERVICE_DESCRIPTION']}")
lines.append("─" * 60)
lines.append("")

for item in review_results:
    lines.append(f"【規則{item['rule_id']}：{item['rule_name']}】")
    lines.append(item["result"])
    lines.append("")
    lines.append("─" * 60)
    lines.append("")

lines.append("【審查說明】")
lines.append("本報告由 AI 輔助生成，僅供參考。")
lines.append("最終契約決策仍應由具備法律專業資格之人員確認。")
lines.append("=" * 60)

report_text = "\n".join(lines)
print(report_text)

filename = f"contract_review_report_{datetime.now().strftime('%Y%m%d_%H%M')}.txt"
with open(filename, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"\n💾 報告已儲存為：{filename}")
print("📥 請到左側檔案區（資料夾圖示）下載此報告檔案")
print("✅ 完整流程執行完畢！")

## 🎓 延伸練習

恭喜完成本教材！以下是三個進階挑戰：

---

### 🔹 練習一：比較三個 AI 的差異
回到 Step 2，把 `AI_MODEL` 分別改成 `"gemini"`、`"gpt"`、`"claude"`，各執行一次，比較三個模型的審查結果有何不同。

---

### 🔹 練習二：新增第六條審查規則
在 Step 6 的 `REVIEW_RULES` 列表中，仿照現有規則格式新增第六條：
- 規則名稱：「爭議解決機制」
- 審查重點：管轄法院是否對乙方方便？是否應加入仲裁條款？

---

### 🔹 練習三：修改當事人資料
前往 GitHub 上的 `party_data.json`，把違約金比例改成 `"8"`（千分之八），重新執行 Step 7～8，觀察審查結果有何不同。

---

> 💡 **思考問題**：三個 AI 模型的審查結果有什麼差異？哪個模型的建議對法務工作最有幫助？為什麼？
